**[Course Home](../README.md) | [Python Lab](../python/00_START_HERE.ipynb) | [Next: Ch 01: Conservation of Flow](01_little_law_and_conservation_of_flow.ipynb)**

*Part I: Foundations of Flow & Memoryless Systems — Chapter 00 of 11*

---

# Chapter 0: The Big Picture — Why Average Utilization Lies & The Calculus of Congestion
### *The Illusion of Spare Capacity, The Flaw of Averages, and The Non-Linear Mathematics of Queueing Delays*

---

## 1. The Core Engineering Problem: Why Average Utilization Lies

In production infrastructure engineering, capacity planning is often driven by a dangerously misleading intuition:

> *"Our hardware testbeds and CI/CD runner pools are only running at 85% average CPU and worker utilization. Therefore, we have 15% spare capacity, and jobs should experience virtually zero waiting time."*

This intuition is fatally wrong. When systems experience stochastic variability—meaning job arrivals occur irregularly and job runtimes vary—average utilization is not a buffer; it is a fuse.

In deterministic systems (such as a factory assembly line where a part arrives every 60 seconds and takes exactly 50 seconds to mill), 85% utilization produces a clean, empty queue between cycles. But in computing networks, cloud worker pools, and automated test fleets:
1. **Arrivals are bursty**: Developers push pull requests (PRs) in waves following code reviews, automated bots trigger multi-target test matrices simultaneously, and webhook events fire in clusters.
2. **Execution times vary widely**: Fast unit test targets finish in 4 seconds; end-to-end integration and hardware-in-the-loop test runs take 45 minutes.

When random arrivals collide with varying service durations, temporary arrival surges inevitably exceed instantaneous service capacity. Backlog accumulates. Because a server cannot "pre-serve" future jobs during idle intervals, the capacity lost during an idle period is permanently forfeit. When the next burst arrives, the backlog cannot be processed instantaneously.

The mathematical consequence is profound: **queueing delay is a strictly convex, non-linear function of utilization**. As utilization approaches 100%, queue length and waiting time do not increase linearly—they explode toward infinity according to the hyperbolic factor:

$$
L \propto \frac{\rho}{1 - \rho}
$$

where $\rho$ is the traffic intensity (utilization). 

```
Average Queue Delay W_q
    ^
    |                                                * (Asymptote at rho = 1.0)
    |                                               *
    |                                              *
    |                                             *
    |                                           *
    |                                         *
    |                                      *
    |                             *   *
    |                 *    *
    |    *    *
    +----------------------------------------------------> Traffic Intensity rho
        0.1   0.2   0.3   0.4   0.5   0.6   0.7   0.8   0.9  1.0
        [--- Safe Linear Zone ---]   [-- Hockey Stick --]  [ Catastrophic ]
```

At 50% utilization, an engineer experiences minimal queueing. At 80% utilization, the average queue length quadruples. At 95% utilization, the backlog is nearly twenty times larger. At 99% utilization, queueing delay dominates total execution time by orders of magnitude.

Understanding this calculus is what separates naive infrastructure guessing from rigorous performance engineering.

---

## 2. What Queueing Theory Allows Engineers to Do

Queueing theory and stochastic renewal processes provide the analytical tools required to design, dimension, and operate distributed systems under uncertainty. Specifically, this mathematical framework enables engineers to:

1. **Calculate the Exact Latency-Utilization Trade-Off**: Determine the exact inflection point beyond which adding load causes catastrophic queue collapse, establishing empirical operational limits for auto-scaling policies.
2. **Dimension Fleet Size for Tail-Latency SLAs**: Compute the minimum number of worker nodes, hardware-in-the-loop fixtures, or database connection slots required to guarantee that 95% or 99% of requests complete within an agreed service-level objective (SLO).
3. **Quantify the ROI of Variance Reduction**: Mathematically prove whether spending engineering effort to split long-running tests into uniform chunks yields a higher latency reduction than purchasing 50% more servers (often it does, via the Pollaczek-Khinchine relation).
4. **Design Preemptive and Non-Preemptive Schedulers**: Establish optimal dispatching policies (such as Shortest Remaining Processing Time or the $c\mu$-rule) that minimize mean cycle time without starving high-priority workloads.
5. **Architect Resilient Networks of Queues**: Identify throughput bottlenecks across multi-stage compilation and test pipelines, preventing upstream stages from overwhelming downstream merge queues.

---

## 3. End-to-End Engineering Anchor Scenario: The CI/CD Test Fleet Bottleneck

To keep every derivation grounded in operational reality, this course will reference a concrete, end-to-end engineering scenario: **an enterprise CI/CD verification fleet**.

### The System Architecture

Consider an automated software verification pipeline responsible for gating pull requests before they are merged into the main trunk.

```
+------------------+      Webhook Event      +--------------------+
| Developer PRs    | ----------------------> | Global Merge Queue |
| (Poisson Stream) |                         | (FIFO Buffer)      |
+------------------+                         +--------------------+
                                                       |
                                      Dispatch Request |
                                                       v
                                             +--------------------+
                                             | Dispatcher Router  |
                                             +--------------------+
                                              /        |         \
                                             /         |          \
                                            v          v           v
                                       +--------+  +--------+  +--------+
                                       | Testbed|  | Testbed|  | Testbed|
                                       | Node 1 |  | Node 2 |  | Node c |
                                       +--------+  +--------+  +--------+
                                            \          |          /
                                             \         |         /
                                              v        v        v
                                        +---------------------------+
                                        | Pass/Fail Status Callback |
                                        +---------------------------+
```

* **Arrivals**: Pull requests arrive randomly throughout the working day. During peak hours, webhooks arrive at an average rate of $\lambda = 12$ jobs per hour ($0.2$ jobs/minute).
* **Processing Pool**: A fleet of hardware-in-the-loop testbed runners execute firmware flashing, integration tests, and performance benchmarks.
* **Service Times**: The mean test execution duration is $\mathbb{E}[S] = 4.0$ minutes. The service capacity per runner is therefore $\mu = 1 / 4.0 = 0.25$ jobs per minute ($15$ jobs/hour).

### The Management Paradox

The infrastructure team reports:
* Total fleet hardware utilization: $\rho = 80\%$.
* Executive assumption: "The fleet has 20% unused headroom. Turnaround time should equal the raw execution time (4 minutes) plus maybe a few seconds of queueing."

The reality observed by software developers:
* Average turnaround time is **20 minutes** (16 minutes spent waiting in line, 4 minutes running).
* The 95th-percentile turnaround time exceeds **60 minutes**.
* Developers report that the merge queue feels "completely frozen."

Why does 20% spare capacity produce a 400% latency penalty on average, and a 1,400% penalty at the tail? This course will demonstrate the exact mathematical proof.

---

## 4. Fundamental Definitions: The Triad of Flow

Every queueing system is characterized by three fundamental parameters:

### 1. Arrival Rate ($\lambda$)
The mean number of incoming requests per unit time:

$$
\lambda = \lim_{t \to \infty} \frac{A(t)}{t}
$$

where $A(t)$ is the cumulative count of arrivals in the interval $[0, t]$. The mean inter-arrival time is:

$$
\mathbb{E}[\tau] = \frac{1}{\lambda}
$$

### 2. Service Rate ($\mu$) and Service Duration ($S$)
The mean duration required to process a single job by a dedicated server is denoted $S$, with expected value:

$$
\mathbb{E}[S] = \frac{1}{\mu}
$$

where $\mu$ is the service capacity (jobs processed per unit time when the server is continuously busy).

### 3. Traffic Intensity and Server Utilization ($\rho$)
For a single server ($c = 1$), traffic intensity is defined as the ratio of arrival rate to service capacity:

$$
\rho = \frac{\lambda}{\mu} = \lambda \mathbb{E}[S]
$$

For a multi-server pool with $c$ identical parallel servers:

$$
\rho = \frac{\lambda}{c \mu} = \frac{\lambda \mathbb{E}[S]}{c}
$$

### The Stability Criterion
A queueing system is **linearly stable** if and only if the long-term arrival rate is strictly less than the aggregate maximum processing capacity:

$$
\rho < 1.0 \iff \lambda < c \mu
$$

* If $\rho < 1$: The system reaches a stationary, ergodic steady-state distribution. Queue lengths fluctuate around a finite mean.
* If $\rho = 1$: The queue behaves like an unbiased random walk. Queue length variance grows linearly with time, and average waiting time diverges to infinity.
* If $\rho > 1$: The system is unstable. Jobs arrive faster than they can be cleared. The queue backlog grows deterministically at rate $\lambda - c\mu$, and latency grows without bound.

---

## 5. Physical Mental Model: The Asymmetry of Idle Time

Why is queue growth non-linear? Why does an 80% loaded system experience noticeable queues, while an 85% loaded system experiences severe delay?

Consider a single server operating over a timeline of discrete time slots:

```
Timeline:    t0   t1   t2   t3   t4   t5   t6   t7   t8   t9  t10  t11  t12
Arrivals:    .    .    ARR  .    .    ARR  ARR  ARR  .    .    .    .    .
Server:     IDLE IDLE BUSY IDLE IDLE BUSY BUSY BUSY BUSY .    .    .    .
Queue:       0    0    0    0    0    0    1    2    1    0    0    0    0
                      ^                   ^
               Idle capacity       Burst causes queue
               wasted forever      to accumulate
```

1. **Capacity Cannot Be Stored**: At times $t_0, t_1, t_3, t_4$, the server is idle. The runner could process jobs, but no jobs are present. That compute capacity cannot be banked in a battery or saved for later.
2. **Backlog Accumulates Instantly**: At times $t_5, t_6, t_7$, three jobs arrive in rapid succession. Because the runner can only process one job at a time, jobs immediately stack up in memory or disk buffers.
3. **Asymmetric Clearing Rate**: To clear a backlog of $k$ jobs, the server requires $k$ dedicated service slots. If subsequent arrivals occur before the backlog is fully cleared, the queue compounds.

As utilization $\rho \to 1$, the server is idle less often. When an arrival burst occurs, the probability that the server is already busy approaches 1.0, and the expected time until the server becomes idle again diverges.

---

## 6. The Flaw of Averages in Capacity Planning

Many engineering organizations fall victim to what statistician Sam L. Savage termed **The Flaw of Averages**:

> *Plans based on average assumptions are wrong on average.*

Mathematically, this is Jensen's Inequality. If a function $g(x)$ is strictly convex ($g''(x) > 0$), then:

$$
\mathbb{E}[g(X)] > g(\mathbb{E}[X])
$$

In queueing systems, the function mapping instantaneous load $x$ to waiting time $W_q(x)$ is strictly convex. Consider an infrastructure runner pool subject to load that fluctuates between a slow period and a peak period:
* For 50% of the day, arrival rate is low: $\rho_1 = 0.60$.
* For 50% of the day, arrival rate is high: $\rho_2 = 0.90$.
* The simple arithmetic average utilization is:

$$
\bar{\rho} = \frac{0.60 + 0.90}{2} = 0.75 \quad (75\%)
$$

If an engineer naively evaluates queue delay at the average utilization $\bar{\rho} = 0.75$ using the standard single-server formula $W_q = \frac{\rho}{\mu(1 - \rho)}$ with $\mu = 1$:

$$
W_q(\bar{\rho}) = \frac{0.75}{1 - 0.75} = 3.0 \text{ time units}
$$

Now calculate the true expected waiting time averaged across the day:

$$
\mathbb{E}[W_q] = \frac{1}{2} W_q(\rho_1) + \frac{1}{2} W_q(\rho_2) = \frac{1}{2}\left(\frac{0.60}{1 - 0.60}\right) + \frac{1}{2}\left(\frac{0.90}{1 - 0.90}\right)
$$

$$
\mathbb{E}[W_q] = \frac{1}{2}(1.5) + \frac{1}{2}(9.0) = 0.75 + 4.5 = 5.25 \text{ time units}
$$

The true average waiting time is **75% higher** than the waiting time predicted by the average utilization ($5.25$ vs $3.0$)! 

The surge during the peak period ($\rho = 0.90$) causes an explosion in queue depth ($9.0$) that completely overwhelms the quiet period ($1.5$). The quiet period cannot compensate for the congested period because delay cannot drop below zero.

---

## 7. Concrete Numerical Walkthrough: The Escalation of Congestion

Let us trace the exact numbers for a single-server runner queue ($M/M/1$) where jobs take an average of $\mathbb{E}[S] = 10$ minutes ($\mu = 0.10$ jobs/min).

We vary the arrival rate $\lambda$ from $0.01$ to $0.099$ jobs/min, driving utilization $\rho = \lambda / \mu$ from $10\%$ to $99\%$:

| Utilization $\rho$ | Arrival Rate $\lambda$ (jobs/hr) | Mean Queue Wait $W_q$ (min) | Mean Total Time $W$ (min) | Mean Backlog $L_q$ (jobs) | Total Jobs in System $L$ |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **0.10** | 0.6 | 1.11 | 11.11 | 0.01 | 0.11 |
| **0.30** | 1.8 | 4.29 | 14.29 | 0.13 | 0.43 |
| **0.50** | 3.0 | 10.00 | 20.00 | 0.50 | 1.00 |
| **0.70** | 4.2 | 23.33 | 33.33 | 1.63 | 2.33 |
| **0.80** | 4.8 | 40.00 | 50.00 | 3.20 | 4.00 |
| **0.85** | 5.1 | 56.67 | 66.67 | 4.82 | 5.67 |
| **0.90** | 5.4 | 90.00 | 100.00 | 8.10 | 9.00 |
| **0.95** | 5.7 | 190.00 | 200.00 | 18.05 | 19.00 |
| **0.98** | 5.88 | 490.00 | 500.00 | 48.02 | 49.00 |
| **0.99** | 5.94 | 990.00 | 1000.00 | 98.01 | 99.00 |

### Critical Observations from the Data:
1. **From 50% to 80% Utilization**: Utilization increases by a factor of $1.6\times$. But waiting time in queue increases from $10$ minutes to $40$ minutes—a **$4.0\times$ increase**.
2. **From 80% to 95% Utilization**: Utilization increases by only 15 percentage points. But waiting time leaps from $40$ minutes to $190$ minutes—a **$4.75\times$ increase**.
3. **The 95% Danger Zone**: At $\rho = 0.95$, a developer spends $190$ minutes waiting in line to run a test that takes only $10$ minutes of compute time. The queue wait accounts for **95% of total turnaround time**.

---

## 8. Course Roadmap: The Progression from Flow to Fleet Architecture

This course is organized into three structured pedagogical parts followed by deep-dive technical appendices:

### Part I: Foundations of Flow & Memoryless Systems (Chapters 00–03)
* **Chapter 00: The Big Picture** (This chapter): The illusion of utilization, the calculus of congestion, and the convex mechanics of waiting lines.
* **Chapter 01: Conservation of Flow & Little's Law**: The foundational relation $L = \lambda W$, operational boundaries, throughput bounds, and decomposing systems into queue vs service components.
* **Chapter 02: Poisson Processes & Memoryless Arrivals**: Arrival stream dynamics, exponential inter-arrivals, merging and thinning operations, and PASTA (Poisson Arrivals See Time Averages).
* **Chapter 03: Continuous-Time Markov Chains & Erlang Formulas**: Birth-death processes, detailed balance, $M/M/1$, $M/M/c$, Erlang-C waiting probability, and the statistical economies of scale in server pooling.

### Part II: Renewal Theory, Variance & Heavy Traffic (Chapters 04–06)
* **Chapter 04: Renewal Theory & The Inspection Paradox**: Residual life, size-biased sampling, why random probes catch long jobs, and the Elementary Renewal Theorem.
* **Chapter 05: Service Variance & Pollaczek-Khinchine**: The $M/G/1$ queue, the $C_v^2$ variance penalty, and how heavy-tailed jobs degrade linearly stable systems.
* **Chapter 06: Heavy-Traffic Limits & Kingman's Equation**: Kingman's universal $G/G/c$ approximation, the $\frac{\rho}{1-\rho}$ asymptotic hockey stick, and shadow pricing capacity.

### Part III: Operational Realities & Scheduling Control (Chapters 07–09)
* **Chapter 07: Queues with Server Breakdowns & Vacations**: Server unreliability, repair distributions, setup vacations, and effective capacity degradation.
* **Chapter 08: Scheduling Disciplines & Preemption**: FIFO vs Shortest Job First (SJF), Shortest Remaining Processing Time (SRPT), the $c\mu$-rule, and head-of-line blocking.
* **Chapter 09: Networks of Queues & Bottleneck Routing**: Burke's Theorem, open and closed Jackson networks, bottleneck identification, and load-balanced routing.

### Reference Appendices (Chapters 10–11)
* **Chapter 10: Appendix A — The Queueing Theory Rosetta Stone**: Kendall's notation guide, diagnostic decision trees, formula cheat-sheet, and common traps.
* **Chapter 11: Appendix B — Queueing Theory of Flaky Tests & CI/CD**: Retry storms, merge queue batching dynamics, and tail latency mitigation.

---

## 9. Hands-On 3-Tier Scaffolding Challenges

### 🟢 Warmup (Tier 1): The SRE Capacity Panic
**Scenario**: An engineering team operates a single dedicated testbed runner. Incoming test requests arrive according to a Poisson process. The runner processes jobs with exponentially distributed runtimes averaging $\mathbb{E}[S] = 12$ minutes ($\mu = 5$ jobs/hour).

1. Initially, the arrival rate is $\lambda = 3.5$ jobs/hour. Calculate server utilization $\rho$, average queue wait time $W_q$ (in minutes), and average number of jobs in the queue $L_q$.
2. The arrival rate increases by just 1 job/hour to $\lambda = 4.5$ jobs/hour. Calculate the new $\rho$, $W_q$, and $L_q$.
3. Compute the percentage increase in arrival rate versus the percentage increase in queue wait time.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Initial State ($\lambda = 3.5$, $\mu = 5.0$)**:
   $$\rho_1 = \frac{3.5}{5.0} = 0.70 \quad (70\%)$$
   $$W_q = \frac{\rho}{\mu(1 - \rho)} = \frac{0.70}{5.0(1 - 0.70)} = \frac{0.70}{1.50} \approx 0.4667 \text{ hours} = \mathbf{28.0 \text{ minutes}}$$
   $$L_q = \lambda W_q = 3.5 \times 0.4667 \approx \mathbf{1.633 \text{ jobs}}$$

2. **Loaded State ($\lambda = 4.5$, $\mu = 5.0$)**:
   $$\rho_2 = \frac{4.5}{5.0} = 0.90 \quad (90\%)$$
   $$W_q = \frac{0.90}{5.0(1 - 0.90)} = \frac{0.90}{0.50} = 1.80 \text{ hours} = \mathbf{108.0 \text{ minutes}}$$
   $$L_q = \lambda W_q = 4.5 \times 1.80 = \mathbf{8.10 \text{ jobs}}$$

3. **Comparison**:
   * Percentage increase in arrival rate:
     $$\frac{4.5 - 3.5}{3.5} \times 100\% = \frac{1.0}{3.5} \times 100\% = \mathbf{+28.57\%}$$
   * Percentage increase in queue wait time:
     $$\frac{108.0 - 28.0}{28.0} \times 100\% = \frac{80.0}{28.0} \times 100\% = \mathbf{+285.7\%}$$

**Key Engineering Takeaway**: A modest $28.6\%$ increase in traffic produces an order-of-magnitude ($285.7\%$) explosion in delay because the system was pushed deep into the convex hockey-stick regime ($\rho = 0.70 \to 0.90$).
</details>

---

### 🟡 Diagnostic (Tier 2): Evaluating Time-Varying Load via Jensen's Inequality
**Scenario**: A build farm experiences a bimodal load profile across a 12-hour workday:
* **Morning quiet period** (6 hours): $\lambda_1 = 6$ builds/hour.
* **Afternoon rush period** (6 hours): $\lambda_2 = 9$ builds/hour.
The build cluster has an effective processing capacity of $\mu = 10$ builds/hour.

1. What is the naive unweighted average arrival rate $\bar{\lambda}$, and what queue wait time $W_q(\bar{\lambda})$ would a capacity planner predict using this average?
2. What is the actual average queue wait time $\bar{W}_q = \frac{1}{2} W_q(\lambda_1) + \frac{1}{2} W_q(\lambda_2)$ experienced by developers across the 12 hours?
3. What is the error percentage introduced by relying on average utilization?

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Naive Prediction**:
   $$\bar{\lambda} = \frac{6 + 9}{2} = 7.5 \text{ builds/hour}$$
   $$\bar{\rho} = \frac{7.5}{10} = 0.75 \quad (75\%)$$
   $$W_q(\bar{\lambda}) = \frac{\bar{\rho}}{\mu(1 - \bar{\rho})} = \frac{0.75}{10(1 - 0.75)} = \frac{0.75}{2.50} = 0.30 \text{ hours} = \mathbf{18.0 \text{ minutes}}$$

2. **Actual Experienced Mean Waiting Time**:
   * For the morning period ($\lambda_1 = 6, \rho_1 = 0.60$):
     $$W_{q,1} = \frac{0.60}{10(1 - 0.60)} = \frac{0.60}{4.0} = 0.15 \text{ hours} = 9.0 \text{ minutes}$$
   * For the afternoon period ($\lambda_2 = 9, \rho_2 = 0.90$):
     $$W_{q,2} = \frac{0.90}{10(1 - 0.90)} = \frac{0.90}{1.0} = 0.90 \text{ hours} = 54.0 \text{ minutes}$$
   * Weighted average wait time over the 12 hours:
     $$\bar{W}_q = \frac{6 \times 9.0 + 6 \times 54.0}{12} = \frac{9.0 + 54.0}{2} = \frac{63.0}{2} = \mathbf{31.5 \text{ minutes}}$$

3. **Percentage Error**:
   $$\text{Error} = \frac{\bar{W}_q - W_q(\bar{\lambda})}{W_q(\bar{\lambda})} \times 100\% = \frac{31.5 - 18.0}{18.0} \times 100\% = \mathbf{+75.0\%}$$

**Key Engineering Takeaway**: By ignoring peak-period variation and computing latency from average utilization, the capacity planner underestimates actual developer waiting time by 75%. In real fleets with sharper peak-to-trough ratios, the error frequently exceeds $300\%$.
</details>

---

### 🔴 Capstone Challenge (Tier 3): Sizing an On-Demand Test Fleet for SLA Compliance
**Scenario**: You are tasked with provisioning cloud worker instances for a high-priority integration test suite.
* Pull requests arrive at rate $\lambda = 30$ PRs/hour.
* Each PR runs an integration suite with mean duration $\mathbb{E}[S] = 6$ minutes ($\mu = 10$ PRs/hour per worker).
* The SLA requires that the **mean queue waiting time must not exceed 2 minutes** ($W_q \le 2/60 = 0.0333$ hours).

Assume that PRs can be dispatched across a pool of $c$ independent single-worker queues (or an approximation where each server handles an equal share $\lambda / c$).
1. If the workload were split evenly across $c$ independent workers, find the minimum integer number of workers $c$ required such that each worker's queue wait satisfies $W_q \le 2$ minutes.
2. For this fleet size $c$, what is the resulting average server utilization $\rho$?
3. If management demands $\rho \ge 85\%$ utilization to minimize cloud billing, can the 2-minute SLA be satisfied with single-server partitioning? What is the maximum permissible utilization?

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Minimum Workers under Decentralized Dispatch**:
   If each worker receives an arrival stream of $\lambda_i = \lambda / c = 30 / c$, and has service capacity $\mu = 10$:
   $$W_{q, i} = \frac{\lambda_i}{\mu(\mu - \lambda_i)} = \frac{30/c}{10(10 - 30/c)} \le \frac{2}{60} = \frac{1}{30}$$
   Multiply both sides by 30:
   $$\frac{900/c}{10(10 - 30/c)} \le 1 \implies \frac{90/c}{10 - 30/c} \le 1$$
   Multiply by $(10 - 30/c)$ (since $\mu > \lambda_i \implies 10 - 30/c > 0$):
   $$\frac{90}{c} \le 10 - \frac{30}{c} \implies \frac{120}{c} \le 10 \implies c \ge 12$$

   We require at least **$c = 12$ workers**.

2. **Resulting Utilization**:
   $$\lambda_i = \frac{30}{12} = 2.5 \text{ PRs/hour per worker}$$
   $$\rho = \frac{\lambda_i}{\mu} = \frac{2.5}{10} = \mathbf{0.25 \quad (25\%)}$$
   Check wait time at $c = 12$:
   $$W_q = \frac{0.25}{10(1 - 0.25)} = \frac{0.25}{7.5} = \frac{1}{30} \text{ hours} = 2.0 \text{ minutes}$$

3. **Management Constraint ($\rho \ge 0.85$)**:
   If utilization is $\rho = 0.85$:
   $$W_q = \frac{0.85}{10(1 - 0.85)} = \frac{0.85}{1.5} = 0.5667 \text{ hours} = \mathbf{34.0 \text{ minutes}}$$
   This is **17 times higher** than the 2-minute SLA!
   
   To satisfy $W_q \le 2$ minutes:
   $$\frac{\rho}{10(1 - \rho)} \le \frac{1}{30} \implies \frac{\rho}{1 - \rho} \le \frac{1}{3} \implies 3\rho \le 1 - \rho \implies 4\rho \le 1 \implies \rho \le 0.25$$
   The maximum permissible server utilization is **$25\%$**. Any utilization above $25\%$ strictly violates the 2-minute SLA under partitioned single-server queues!

**Key Engineering Takeaway**: Strict tail and mean latency SLAs in stochastic environments require significant apparent "over-provisioning" when queues are partitioned. (In Chapter 03, we will see how centralized multi-server pooling ($M/M/c$) dramatically improves this trade-off).
</details>

---

**[Course Home](../README.md) | [Python Lab](../python/00_START_HERE.ipynb) | [Next: Ch 01: Conservation of Flow](01_little_law_and_conservation_of_flow.ipynb)**
